### Soil Photos DataSets

* To analyze our photos, we are required to scrape them from the Soil Of Africa Website. 
* The main challenge is that the website is slow to load and downloading of the full dataset is challenging.
* We therefore created a function to scrape to aid with scraping the Website for photos and download in batches:

#### Loading the Soil Photos Data
We developed a scraping function that:
1. Opens the Soil Data of Africa dashboard (ISRIC / Soils4Africa) in an automated Chrome browser using Selenium.
2. Configures the browser to automatically download files into a specified local folder without prompting.
3. Scrolls to the bottom of the page to ensure all dynamic soil data tables fully load.
4. Waits for the dashboard to finish rendering data before interacting with it.
5. Locates the table control and changes the display setting to show 20 entries per page.
6. Waits again to allow the expanded dataset (more soil records per page) to load completely.
7. Repeats a loop across all pages of the dataset.
8. On each page, clicks the table’s options (3-dot menu) to access export functions.
9. Selects the Download → Export to CSV option to retrieve the current page’s data.
10. Triggers download of a ZIP file containing the CSV dataset for that page.
11. Monitors the download process and waits until the file is fully completed.
12. Adds extra buffer time to ensure the file is properly written to disk.
13. Identifies the most recently downloaded ZIP file in the directory.
14. Renames each file sequentially (e.g., K1.zip, K2.zip, …) for organization and order.
15. Retries renaming if the file is temporarily locked by the system (e.g., antivirus interference).
16. Navigates to the next page using the pagination controls on the dashboard.
17. Waits for the next page of soil data to fully load before continuing.
18. Continues this process until all pages have been processed.
19. Handles errors gracefully and stops execution if something goes wrong.
20. Closes the browser session after completion
21. Produces a complete collection of ZIP files containing CSV data, representing the full soil dataset from the dashboard (which may include soil measurements, locations, and possibly references to soil photos rather than the images themselves).

The python file containing the scraping function is in the data folder.
We were able to successfully scrape 21 entries, each with 20 images of Kenyan Soil.

In [1]:
# Installing the necessary libraries
!pip install selenium webdriver-manager

  Using cached selenium-4.43.0-py3-none-any.whl.metadata (7.5 kB)
  Using cached webdriver_manager-4.0.2-py2.py3-none-any.whl.metadata (12 kB)
  Using cached certifi-2026.4.22-py3-none-any.whl.metadata (2.5 kB)
  Using cached trio-0.33.0-py3-none-any.whl.metadata (8.5 kB)
  Using cached trio_websocket-0.12.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached outcome-1.3.0.post0-py2.py3-none-any.whl.metadata (2.6 kB)
  Using cached wsproto-1.3.2-py3-none-any.whl.metadata (5.2 kB)
Using cached selenium-4.43.0-py3-none-any.whl (9.6 MB)

  Attempting uninstall: urllib3

    Found existing installation: urllib3 2.3.0

    Uninstalling urllib3-2.3.0:

      Successfully uninstalled urllib3-2.3.0

   ---- ----------------------------------- 1/9 [urllib3]
   ---- ----------------------------------- 1/9 [urllib3]
  Attempting uninstall: typing_extensions
   -

In [2]:
# importing required libraries for scraping.
import os
import glob
import time
import zipfile
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager

In [5]:
# --- Configuration ---
DOWNLOAD_DIR = r"E:\Batches of Soil Photos\Batches of Soil Photos"
os.makedirs(DOWNLOAD_DIR, exist_ok=True) # Ensures the folder actually exists
TOTAL_PAGES = 21

# 1. Set up the Chrome WebDriver
options = webdriver.ChromeOptions()
prefs = {
    "download.default_directory": DOWNLOAD_DIR,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True
}
options.add_experimental_option("prefs", prefs)

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 180)
actions = ActionChains(driver)

url = "https://dashboards.isric.org/superset/dashboard/46c3bdf3-e215-4f05-b479-60c644c368dd/?permalink_key=AWEgppXGgzd#TAB-_HKoBS_QI"
driver.get(url)

print("Scrolling to the bottom of the page to load all tables...")
# Scroll down by the total height of the body
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

# Wait a few seconds for the bottom table to fetch its data from the server
print("Waiting for bottom table to render...")
time.sleep(60)

try:
    # --- NEW: STEP 1: FILTER FOR KENYA (ACCORDION FIX EDITION) ---
    print("Expanding the 'Country name' filter panel...")
    time.sleep(5) # Let the dashboard settle
    
    # 1. Expand the accordion!
    header_xpath = "//*[contains(text(), 'Country name')]"
    try:
        header = wait.until(EC.element_to_be_clickable((By.XPATH, header_xpath)))
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", header)
        time.sleep(1)
        
        # Click it to slide the panel open
        driver.execute_script("arguments[0].click();", header)
        print("Clicked the accordion header. Waiting for it to slide open...")
        time.sleep(2) # Give the CSS animation time to finish
    except Exception as e:
        print("Warning: Could not click header. If it's already open, we'll continue anyway.")

    # 2. Now that the panel is open, find the input field
    print("Locating the input field...")
    input_xpath = "//*[contains(text(), 'Country name')]/following::input[1]"
    country_input = wait.until(EC.presence_of_element_located((By.XPATH, input_xpath)))

    print("Activating input field...")
    driver.execute_script("arguments[0].click();", country_input)
    time.sleep(1)

    print("Typing 'Kenya'...")
    country_input.send_keys("Kenya")
    
    # Let the React search filter finish its animation before we start looking!
    time.sleep(3)

    # 3. Wait for the Ant Design dropdown option in the portal and click it
    print("Waiting for dropdown option to render...")
    kenya_option_xpath = "//div[contains(@class, 'ant-select-item-option-content') and text()='Kenya']"
    kenya_option = wait.until(EC.presence_of_element_located((By.XPATH, kenya_option_xpath)))
    
    # JS click is incredibly safe for floating dropdown menus
    driver.execute_script("arguments[0].click();", kenya_option)
    time.sleep(2)

    # 4. Apply filters
    print("Clicking Apply filters...")
    apply_button = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "button.filter-apply-button")))
    driver.execute_script("arguments[0].click();", apply_button)

    print("Waiting 60 seconds for the dashboard to reload with Kenya's data...")
    time.sleep(60)

    # --- STEP 2: Change to 20 Entries ---
    print("Adjusting entries to 20...")
    dropdown_element = wait.until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, "#chart-id-290 div.form-inline.dt-controls select")
    ))
    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", dropdown_element)
    time.sleep(5)
    
    Select(dropdown_element).select_by_visible_text('20')
    print("Waiting 60 seconds for the table to reload with 20 rows...")
    time.sleep(60)

    # --- STEP 3: The Main Scraping Loop ---
    for current_page in range(1, TOTAL_PAGES + 1):
        print(f"--- Processing Page {current_page} of {TOTAL_PAGES} ---")
        
        # A. Click the BOTTOM-MOST 3-dots menu on the page
        print("Clicking the 3-dots menu for the bottom table...")
        js_click_bottom_menu = """
        let menus = document.querySelectorAll('[id$="-controls"] > div, .fa-ellipsis-v');
        if(menus.length > 0) {
            let targetMenu = menus[menus.length - 1]; // Grab the last one
            targetMenu.scrollIntoView({block: 'center'});
            targetMenu.click();
            return true;
        }
        return false;
        """
        driver.execute_script(js_click_bottom_menu)
        time.sleep(3)

        # B. Hover over "Download"
        download_menu_item = wait.until(EC.presence_of_element_located(
            (By.XPATH, "//div[@title='Download']")
        ))
        actions.move_to_element(download_menu_item).perform()
        time.sleep(3) 

        # C. Click the LAST "Export to .CSV" button
        print("Clicking Export to CSV...")
        js_export_click = """
        let items = Array.from(document.querySelectorAll('li, div')).filter(item => 
            item.innerText && item.innerText.includes('Export to .CSV') && item.offsetParent !== null
        );
        if(items.length > 0) {
            items[items.length - 1].click(); // Click the last visible one
            return true;
        }
        return false;
        """
        driver.execute_script(js_export_click)
        
        # D. Download handling and renaming
        print("Downloading ZIP file...")
        while True:
            if not glob.glob(os.path.join(DOWNLOAD_DIR, '*.crdownload')):
                break
            time.sleep(2) 
            
        time.sleep(10) # Buffer
        
        # Rename the downloaded file
        zip_files = glob.glob(os.path.join(DOWNLOAD_DIR, '*.zip'))
        if zip_files:
            latest_file = max(zip_files, key=os.path.getctime)
            new_filename = os.path.join(DOWNLOAD_DIR, f"K{current_page}.zip")
            
            renamed = False
            for attempt in range(5): 
                try:
                    if os.path.exists(new_filename):
                        os.remove(new_filename) 
                    os.rename(latest_file, new_filename)
                    print(f"Successfully saved: K{current_page}.zip")
                    renamed = True
                    break 
                except PermissionError:
                    print(f"File locked by Windows. Retrying... (Attempt {attempt + 1}/5)")
                    time.sleep(2)
            if not renamed:
                print(f"WARNING: Could not rename {latest_file}.")
        else:
            print(f"WARNING: ZIP file not found for page {current_page}!")
            
        # E. Click the next page number (THE "GREY HIGHLIGHT" FIX)
        if current_page == TOTAL_PAGES:
            print("Successfully scraped all pages for Kenya!")
            break
            
        next_page_num = str(current_page + 1)
        print(f"Scrolling to bottom table to navigate to page {next_page_num}...")
        
        try:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2) 
            
            # 1. Target the specific Ant Design pagination button using its 'title' attribute
            btn_xpath = f"(//ul[contains(@class, 'ant-pagination') or contains(@class, 'pagination')])[last()]//li[@title='{next_page_num}']"
            next_page_btn = wait.until(EC.presence_of_element_located((By.XPATH, btn_xpath)))
            
            driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", next_page_btn)
            time.sleep(1)
            
            # 2. Click the <a> tag INSIDE the <li>, which is the true React event listener
            print(f"Clicking page {next_page_num}...")
            js_click_exact = """
            let li = arguments[0];
            let a = li.querySelector('a');
            if (a) { 
                a.click(); 
            } else { 
                li.click(); 
            }
            """
            driver.execute_script(js_click_exact, next_page_btn)
            
            # 3. VERIFY THE CLICK WORKED (Look for the grey highlight!)
            print("Waiting for the button to turn grey (active)...")
            
            active_btn_xpath = f"(//ul[contains(@class, 'ant-pagination') or contains(@class, 'pagination')])[last()]//li[contains(@class, 'ant-pagination-item-active') and @title='{next_page_num}']"
            
            # Wait up to 15 seconds for the CSS to change to 'active'
            WebDriverWait(driver, 15).until(EC.presence_of_element_located((By.XPATH, active_btn_xpath)))
            print(f"✅ Success! Page {next_page_num} is now highlighted in grey.")
            
            print("Waiting 30 seconds for the new table rows to fetch from the server...")
            time.sleep(30)
            
        except Exception as e:
            debug_img_path = os.path.join(DOWNLOAD_DIR, f"pagination_error_page_{next_page_num}.png")
            driver.save_screenshot(debug_img_path)
            print(f"\n❌ CRITICAL: Page {next_page_num} did not turn grey!")
            raise Exception(f"Pagination failed on page {current_page}. The button didn't activate. See screenshot {debug_img_path}")

# Added the missing outer 'except' block back in!
except Exception as e:
    print(f"\nScraper stopped due to a critical error: {e}")

finally:
    driver.quit()
    print("Browser session closed.")
    
# --- FINAL STEP: Extract and combine all scraped batches into one file ---
all_dataframes = []

zip_files = glob.glob(os.path.join(DOWNLOAD_DIR, 'K*.zip'))
print(f"Found {len(zip_files)} zip files. Extracting and combining...")

for zip_path in zip_files:
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            csv_files = [f for f in z.namelist() if f.endswith('.csv')]
            for csv_file in csv_files:
                with z.open(csv_file) as f:
                    df = pd.read_csv(f)
                    all_dataframes.append(df)
    except Exception as e:
        print(f"Error reading {zip_path}: {e}")

if all_dataframes:
    final_combined_df = pd.concat(all_dataframes, ignore_index=True)
    
    master_file_path = os.path.join(DOWNLOAD_DIR, "all_scraped_soil_photos.csv")
    final_combined_df.to_csv(master_file_path, index=False)
    
    print(f"✅ Success! All data combined and saved to: {master_file_path}")
    print(f"Total rows gathered: {len(final_combined_df)}")
else:
    print("❌ No CSV data found to combine. Check the downloaded files.")

ReadTimeoutError: HTTPConnectionPool(host='localhost', port=63528): Read timed out. (read timeout=120)

In [ ]:
# Now that we have confirmed that we can view our images once downloaded, let us download all our 21 datasets and concatenate our rows.
# We expect a total of 400 entries after the concatenation.

soil_photos_df = []

for i in range(1, 21):
    file = f"scraped_datasets/K{i}.csv"
    df = pd.read_csv(file)
    soil_photos_df.append(df)

combined_soil_photos_df = pd.concat(soil_photos_df, ignore_index=True)

#saving the combined dataset to a new CSV file
combined_soil_photos_df.to_csv("combined_soil_photos.csv", index=False)